In [ ]:
!pip install pandas numpy torch transformers scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support


In [ ]:
# ✅ Load dataset from CSV file
df = pd.read_csv("Modified_Bengali_Review_Dataset.csv")

In [ ]:
def preprocess_data(df):
    # No need to rename columns or map labels since they are already 0 and 1
    #df = df.rename(columns={"Reviews": "Review", "Sentiment": "Label"})

    # ✅ Remove missing values
    df.dropna(inplace=True)
    return df

In [ ]:
print(df.head())

                              Reviews  Sentiment
0     অসাধারণ নিশো বস্ আর অমি ভাইকেও।          0
1   "এত মোটা বাশ নিতে পারছি না বাবা "          1
2                  নাটক আসলেই অসাধারণ          0
3                     ফালতু একটা নাটক          1
4         ধুমপান সাস্থর জন্য ক্ষতিকর।          1


In [ ]:
# ✅ Split dataset into train and test sets
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["Reviews"].tolist(), df["Sentiment"].tolist(), test_size=0.2, random_state=42
)

In [ ]:
# ✅ Load BanglaBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("csebuetnlp/banglabert")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/528k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
def tokenize_function(texts, labels):
    tokenized_data = tokenizer(texts, padding="max_length", truncation=True, max_length=256)
    tokenized_data["labels"] = labels
    return tokenized_data


In [ ]:
# ✅ Tokenize dataset
tokenized_train = tokenize_function(train_texts, train_labels)
tokenized_test = tokenize_function(test_texts, test_labels)

In [ ]:
 #✅ Convert to PyTorch tensors using Dataset class
class BengaliReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


In [ ]:
# Create dataset objects
train_dataset = BengaliReviewDataset(tokenized_train, train_labels)
test_dataset = BengaliReviewDataset(tokenized_test, test_labels)


In [ ]:

# ✅ Load pre-trained BanglaBERT model for classification
model = AutoModelForSequenceClassification.from_pretrained("csebuetnlp/banglabert", num_labels=2)


pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# ✅ Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    fp16=True,
)

# ✅ Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-29-76ef07beda02>:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# ✅ Train the model
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: zarin-cse-20210104111 (zarin-cse-20210104111-ahsanullah-university-of-science-t) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,0.129200,0.191850
2,0.001800,0.170293
3,0.001000,0.190707


TrainOutput(global_step=7086, training_loss=0.11857577177585264, metrics={'train_runtime': 948.6165, 'train_samples_per_second': 29.87, 'train_steps_per_second': 7.47, 'total_flos': 3727625876812800.0, 'train_loss': 0.11857577177585264, 'epoch': 3.0})

In [ ]:
# ✅ Evaluate the model
def compute_metrics(pred):
    logits, labels = pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary")
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}


In [ ]:
eval_results = trainer.evaluate()
print("Evaluation Results:", eval_results)

Evaluation Results: {'eval_loss': 0.19070690870285034, 'eval_runtime': 17.4335, 'eval_samples_per_second': 135.487, 'eval_steps_per_second': 33.9, 'epoch': 3.0}


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

# ✅ Function to compute accuracy, precision, recall, and F1-score (Fake & Non-Fake Separately)
def compute_metrics(eval_pred):
    logits, labels = eval_pred  # Extract model outputs
    predictions = np.argmax(logits, axis=-1)  # Convert logits to class predictions

    # ✅ Compute accuracy
    acc = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary")

    # ✅ Compute Fake vs Non-Fake metrics separately
    report = classification_report(labels, predictions, target_names=["Positive", "Negative"], digits=4, output_dict=True)

    # Extract per-class F1-scores
    fake_f1 = report["Negative"]["f1-score"]
    non_fake_f1 = report["Positive"]["f1-score"]
    weighted_f1 = report["weighted avg"]["f1-score"]

    return {
        "accuracy": acc,
        "Negative F1-score": fake_f1,
        "Positive F1-score": non_fake_f1,
        "Weighted F1-score": weighted_f1
    }

In [ ]:
# ✅ Run evaluation properly
predictions, labels, _ = trainer.predict(test_dataset)

# ✅ Convert logits to class labels (0 or 1)
predicted_classes = np.argmax(predictions, axis=-1)

# ✅ Compute Accuracy
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(labels, predicted_classes)

# ✅ Print Accuracy
print(f"✅ Model Accuracy: {accuracy:.4f}")

# ✅ Save results as JSON
import json
results = {"Accuracy": accuracy}

with open("evaluation_results.json", "w") as f:
    json.dump(results, f)

# ✅ Print Results
print("📊 Evaluation Results:", results)


✅ Model Accuracy: 0.9716
📊 Evaluation Results: {'Accuracy': 0.9716342082980525}


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import json
import numpy as np

# ✅ Run evaluation properly
predictions, labels, _ = trainer.predict(test_dataset)

# ✅ Convert logits to class labels (0 or 1)
predicted_classes = np.argmax(predictions, axis=-1)

# ✅ Compute Accuracy
accuracy = accuracy_score(labels, predicted_classes)

# ✅ Compute Precision, Recall, and F1-score
precision, recall, f1, _ = precision_recall_fscore_support(labels, predicted_classes, average="binary")

# ✅ Print Classification Report
print("📊 Classification Report:")
print(classification_report(labels, predicted_classes, target_names=["Negative", "Positive"], digits=4))

# ✅ Save results as JSON
results = {
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-score": f1
}

with open("evaluation_results.json", "w") as f:
    json.dump(results, f)

# ✅ Print Results
print("✅ Final Evaluation Metrics:")
print(json.dumps(results, indent=4))


📊 Classification Report:
              precision    recall  f1-score   support

    Negative     0.9775    0.9828    0.9801      1682
    Positive     0.9568    0.9441    0.9504       680

    accuracy                         0.9716      2362
   macro avg     0.9672    0.9634    0.9653      2362
weighted avg     0.9716    0.9716    0.9716      2362

✅ Final Evaluation Metrics:
{
    "Accuracy": 0.9716342082980525,
    "Precision": 0.9567809239940388,
    "Recall": 0.9441176470588235,
    "F1-score": 0.9504071058475203
}
